# Exp 4 — Hyperparameter Grid Search (LR × Weight Decay)

4 combinations × 5 epochs each ≈ ~20 minutes total.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve()))  # so 'from utils import *' works
from utils import *
print("utils loaded ✓")

In [ ]:
EXP_DIR  = ROOT / 'outputs' / 'swinv2_small_exp4'
CKPT_DIR_E = EXP_DIR / 'checkpoints'
FIG_DIR_E  = EXP_DIR / 'figures'
for d in (CKPT_DIR_E, FIG_DIR_E): d.mkdir(parents=True, exist_ok=True)

TUNE_LRS           = [3e-5, 5e-5]
TUNE_WEIGHT_DECAYS = [1e-2, 3e-2]
TUNE_EPOCHS        = 5
TUNE_PATIENCE      = 2
print(f'Grid: {len(TUNE_LRS)} LRs × {len(TUNE_WEIGHT_DECAYS)} WDs = {len(TUNE_LRS)*len(TUNE_WEIGHT_DECAYS)} combos')

In [ ]:
train_ds    = DRDataset(train_df, TRAIN_DIR, transform=enhanced_train_transform)
val_ds      = DRDataset(val_df,   VAL_DIR,   transform=val_transform)
t_loader    = make_loader(train_ds, sampler=wrs_sampler)
v_loader    = make_val_loader(val_ds)

In [ ]:
import gc
tune_results = []
print(f"{'#':<4} {'LR':>8} {'WD':>8}  {'Stop Ep':>8} {'Val κ':>8}")
print('─' * 44)
combo = 0
for wd in TUNE_WEIGHT_DECAYS:
    for lr in TUNE_LRS:
        combo += 1; set_seed()
        m  = build_swin_model(pretrained=True).to(device)
        cr = FocalLoss(alpha=class_weights_t, gamma=2.0)
        bp = [p for n,p in m.named_parameters() if 'custom_head' not in n and p.requires_grad]
        hp = [p for n,p in m.named_parameters() if 'custom_head'     in n and p.requires_grad]
        op = torch.optim.AdamW([{'params':bp,'lr':lr*0.1},{'params':hp,'lr':lr}], weight_decay=wd)
        sc_lr = torch.optim.lr_scheduler.CosineAnnealingLR(op, T_max=TUNE_EPOCHS)
        sc_amp = torch.amp.GradScaler('cuda', enabled=use_amp)
        best_k=-1.0; pat=TUNE_PATIENCE; stopped=TUNE_EPOCHS
        for ep in range(1, TUNE_EPOCHS+1):
            run_epoch(t_loader, True,  m, cr, op, sc_amp)
            _,_,_,vk = run_epoch(v_loader, False, m, cr, op, sc_amp)
            sc_lr.step()
            if vk > best_k: best_k=vk; pat=TUNE_PATIENCE
            else:
                pat -= 1
                if pat <= 0: stopped=ep; break
        tune_results.append({'lr':lr,'wd':wd,'val_kappa':best_k,'stopped_at':stopped})
        print(f'{combo:<4} {lr:>8.0e} {wd:>8.0e}  {stopped:>8} {best_k:>8.4f}')
        del m; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
import pandas as _pd, json as _json
df = _pd.DataFrame(tune_results).sort_values('val_kappa', ascending=False)
print('\nResults ranked by val kappa:')
print(df.to_string(index=False))
best = df.iloc[0]
print(f'\nBest: LR={best.lr:.0e}, WD={best.wd:.0e} → val κ={best.val_kappa:.4f}')
print(f'Recommendation: set LR={best.lr:.0e} and WEIGHT_DECAY={best.wd:.0e} in Exp3 for best result.')
with open(EXP_DIR / 'exp4_results.json','w') as f:
    _json.dump({'results': tune_results, 'best_lr': best.lr, 'best_wd': best.wd}, f, indent=2)